# Repository guide: Historical decoder diagnostic; earlier 128-segment validation

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


In [ ]:
# ============================================================
# CELL 1: Mount Drive and define project paths
# Experiment: MMS + Tarifit n-gram language model
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

TRAIN_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "train.csv"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation.csv"
)

print("Project:", PROJECT_ROOT.exists())
print("Train:", TRAIN_CSV.exists())
print("Validation:", VALIDATION_CSV.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: True
Train: True
Validation: True


In [ ]:
# ============================================================
# CELL 2: Inventory Corpus V1.1 training text
# ============================================================

import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)

train_texts = (
    train_df["transcription"]
    .dropna()
    .astype(str)
    .str.strip()
)

train_texts = train_texts[
    train_texts != ""
]

num_segments = len(train_texts)
num_words = sum(
    len(text.split())
    for text in train_texts
)
num_chars = sum(
    len(text)
    for text in train_texts
)

unique_words = set()

for text in train_texts:
    unique_words.update(text.split())

print("Training segments:", num_segments)
print("Total words:", num_words)
print("Unique words:", len(unique_words))
print("Total characters:", num_chars)

print("\nExample transcripts:")
for text in train_texts.head(5):
    print("-", text)

Training segments: 1472
Total words: 36185
Unique words: 3977
Total characters: 186041

Example transcripts:
- rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta
- adlis n umezruy n wexraq n yasuɛ lmasiḥ
- adlis n umezruy n wexraq n yasuɛ lmasiḥ mmis n dawud mmis n ibrahim ibrahim ijja dd isḥaq isḥaq ijja dd yaɛqub yaɛqub ijja dd yahuda d aytmas
- yahuda ijja dd fariṣ d zaraḥ ak d tamar fariṣ ijja dd ḥaṣrun ḥaṣrun ijja dd aram aram ijja dd ɛamminadab ɛamminadab ijja dd naḥcun naḥcun ijja dd salmun salmun ijja dd buɛaz zi raḥab
- buɛaz ijja dd ɛubid ak d raɛut ɛubid ijja dd yassa yassa ijja dd dawud ajeǧid uca dawud ijja dd suliman ak d temɣart n uriya


In [ ]:
# ============================================================
# CELL 3: Find additional potential Tarifit text resources
# ============================================================

SOURCE_TEXT_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "source_text"
)

print("Source text directory exists:", SOURCE_TEXT_DIR.exists())

if SOURCE_TEXT_DIR.exists():

    files = [
        p
        for p in SOURCE_TEXT_DIR.rglob("*")
        if p.is_file()
    ]

    print("Files found:", len(files))

    extensions = {}

    for file in files:
        ext = file.suffix.lower()
        extensions[ext] = extensions.get(ext, 0) + 1

    print("\nFile types:")
    for ext, count in sorted(extensions.items()):
        print(f"{ext or '[no extension]'}: {count}")

    print("\nFiles:")
    for file in files[:100]:
        print(file.relative_to(PROJECT_ROOT))

Source text directory exists: True
Files found: 47

File types:
.gdoc: 1
.html: 44
.txt: 2

Files:
data/raw/source_text/tarifit_info/40-MAT-023.html
data/raw/source_text/tarifit_info/40-MAT-019.html
data/raw/source_text/tarifit_info/41-MRK-014.html
data/raw/source_text/tarifit_info/40-MAT-003.html
data/raw/source_text/tarifit_info/41-MRK-002.html
data/raw/source_text/tarifit_info/40-MAT-015.html
data/raw/source_text/tarifit_info/40-MAT-014.html
data/raw/source_text/tarifit_info/40-MAT-002.html
data/raw/source_text/tarifit_info/41-MRK-003.html
data/raw/source_text/tarifit_info/41-MRK-015.html
data/raw/source_text/tarifit_info/40-MAT-018.html
data/raw/source_text/tarifit_info/40-MAT-022.html
data/raw/source_text/tarifit_info/41-MRK-004.html
data/raw/source_text/tarifit_info/41-MRK-012.html
data/raw/source_text/tarifit_info/40-MAT-005.html
data/raw/source_text/tarifit_info/40-MAT-013.html
data/raw/source_text/tarifit_info/40-MAT-025.html
data/raw/source_text/tarifit_info/41-MRK-008.html
d

In [ ]:
# ============================================================
# CELL 4: Find other potential text resources
# ============================================================

TEXT_EXTENSIONS = {
    ".txt",
    ".csv",
    ".tsv",
    ".json",
    ".html",
    ".htm",
    ".srt",
    ".vtt",
    ".md",
}

potential_text_files = []

for folder_name in ["data/raw", "data/processed", "data/metadata"]:

    folder = PROJECT_ROOT / folder_name

    if not folder.exists():
        continue

    for file in folder.rglob("*"):

        if (
            file.is_file()
            and file.suffix.lower() in TEXT_EXTENSIONS
        ):
            potential_text_files.append(file)

print(
    "Potential text-containing files:",
    len(potential_text_files)
)

for file in potential_text_files:
    print(file.relative_to(PROJECT_ROOT))

Potential text-containing files: 267
data/raw/source_text/tarifit_info/40-MAT-023.html
data/raw/source_text/tarifit_info/40-MAT-019.html
data/raw/source_text/tarifit_info/41-MRK-014.html
data/raw/source_text/tarifit_info/40-MAT-003.html
data/raw/source_text/tarifit_info/41-MRK-002.html
data/raw/source_text/tarifit_info/40-MAT-015.html
data/raw/source_text/tarifit_info/40-MAT-014.html
data/raw/source_text/tarifit_info/40-MAT-002.html
data/raw/source_text/tarifit_info/41-MRK-003.html
data/raw/source_text/tarifit_info/41-MRK-015.html
data/raw/source_text/tarifit_info/40-MAT-018.html
data/raw/source_text/tarifit_info/40-MAT-022.html
data/raw/source_text/tarifit_info/41-MRK-004.html
data/raw/source_text/tarifit_info/41-MRK-012.html
data/raw/source_text/tarifit_info/40-MAT-005.html
data/raw/source_text/tarifit_info/40-MAT-013.html
data/raw/source_text/tarifit_info/40-MAT-025.html
data/raw/source_text/tarifit_info/41-MRK-008.html
data/raw/source_text/tarifit_info/40-MAT-009.html
data/raw/sour

In [ ]:
# ============================================================
# CELL 5: Prepare MMS-compatible Tarifit text for LM training
# ============================================================

import re
import unicodedata

def corpus_to_mms_lm_text(text):
    text = unicodedata.normalize("NFC", str(text)).lower()

    # Corpus V1.1 -> common MMS transcription conventions
    text = text.replace("c", "š")

    # Keep e as e for now.
    # We do NOT blindly convert every e -> ə because MMS
    # contains both e and ə and the correspondence is not 1:1.

    # Remove punctuation
    text = re.sub(r"[!?,.\-]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


lm_texts = [
    corpus_to_mms_lm_text(text)
    for text in train_texts
]

print("LM lines:", len(lm_texts))

print("\nExamples:")
for original, converted in zip(
    train_texts.head(5),
    lm_texts[:5]
):
    print("CORPUS:", original)
    print("LM:    ", converted)
    print()

LM lines: 1472

Examples:
CORPUS: rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta
LM:     rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

CORPUS: adlis n umezruy n wexraq n yasuɛ lmasiḥ
LM:     adlis n umezruy n wexraq n yasuɛ lmasiḥ

CORPUS: adlis n umezruy n wexraq n yasuɛ lmasiḥ mmis n dawud mmis n ibrahim ibrahim ijja dd isḥaq isḥaq ijja dd yaɛqub yaɛqub ijja dd yahuda d aytmas
LM:     adlis n umezruy n wexraq n yasuɛ lmasiḥ mmis n dawud mmis n ibrahim ibrahim ijja dd isḥaq isḥaq ijja dd yaɛqub yaɛqub ijja dd yahuda d aytmas

CORPUS: yahuda ijja dd fariṣ d zaraḥ ak d tamar fariṣ ijja dd ḥaṣrun ḥaṣrun ijja dd aram aram ijja dd ɛamminadab ɛamminadab ijja dd naḥcun naḥcun ijja dd salmun salmun ijja dd buɛaz zi raḥab
LM:     yahuda ijja dd fariṣ d zaraḥ ak d tamar fariṣ ijja dd ḥaṣrun ḥaṣrun ijja dd aram aram ijja dd ɛamminadab ɛamminadab ijja dd naḥšun naḥšun ijja dd salmun salmun ijja dd buɛaz zi raḥab

CORPUS: buɛaz ijja dd ɛubid ak d raɛut ɛubid ijja dd yassa yassa ijja dd dawud ajeǧid uc

In [ ]:
# ============================================================
# CELL 6: Save Tarifit LM training corpus
# ============================================================

LM_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "tarifit_lm"
)

LM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LM_TEXT_PATH = (
    LM_DIR
    / "train_mms_orthography.txt"
)

with open(
    LM_TEXT_PATH,
    "w",
    encoding="utf-8"
) as f:
    for text in lm_texts:
        f.write(text + "\n")

print("Saved:")
print(LM_TEXT_PATH)

print(
    "Lines:",
    sum(
        1
        for _ in open(
            LM_TEXT_PATH,
            encoding="utf-8"
        )
    )
)

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/tarifit_lm/train_mms_orthography.txt
Lines: 1472


In [ ]:
# ============================================================
# CELL 7: Load zero-shot MMS model and inspect decoder labels
# ============================================================

import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

MMS_MODEL_ID = (
    "iukocha/mms-tachebdant-from-tarifit"
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

processor = AutoProcessor.from_pretrained(
    MMS_MODEL_ID
)

model = Wav2Vec2ForCTC.from_pretrained(
    MMS_MODEL_ID
)

model.to(DEVICE)
model.eval()

vocab = processor.tokenizer.get_vocab()

labels = [
    token
    for token, idx in sorted(
        vocab.items(),
        key=lambda x: x[1]
    )
]

print("Model:", MMS_MODEL_ID)
print("Vocabulary size:", len(labels))

for i, label in enumerate(labels):
    print(i, repr(label))

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Model: iukocha/mms-tachebdant-from-tarifit
Vocabulary size: 46
0 '!'
1 ','
2 '-'
3 '.'
4 '?'
5 'a'
6 'b'
7 'd'
8 'e'
9 'f'
10 'g'
11 'h'
12 'i'
13 'j'
14 'k'
15 'l'
16 'm'
17 'n'
18 'p'
19 'q'
20 'r'
21 's'
22 't'
23 'u'
24 'w'
25 'x'
26 'y'
27 'z'
28 'ç'
29 'ā'
30 'č'
31 'ŋ'
32 'š'
33 'ə'
34 'ɛ'
35 'ɣ'
36 'ḍ'
37 'ḥ'
38 'ṣ'
39 'ṭ'
40 '[UNK]'
41 'ẓ'
42 '[PAD]'
43 '<s>'
44 '</s>'
45 ' '


In [ ]:
# ============================================================
# CELL 8: Inspect CTC special-token configuration
# ============================================================

print(
    "PAD token:",
    processor.tokenizer.pad_token,
    processor.tokenizer.pad_token_id
)

print(
    "UNK token:",
    processor.tokenizer.unk_token,
    processor.tokenizer.unk_token_id
)

print(
    "Word delimiter:",
    processor.tokenizer.word_delimiter_token,
    processor.tokenizer.word_delimiter_token_id
)

print(
    "Model pad token ID:",
    model.config.pad_token_id
)

PAD token: [PAD] 42
UNK token: [UNK] 41
Word delimiter: | 41
Model pad token ID: 42


In [ ]:
# ============================================================
# CELL 9: Install CTC beam-search + KenLM dependencies
# ============================================================

!pip install -q pyctcdecode kenlm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.5/427.5 kB 15.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 107.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for kenlm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for kenlm
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (kenlm)


In [ ]:
# ============================================================
# CELL 10: Prepare pyctcdecode labels safely
# ============================================================

vocab = processor.tokenizer.get_vocab()

decoder_labels = [
    token
    for token, idx in sorted(
        vocab.items(),
        key=lambda x: x[1]
    )
]

# Actual CTC blank
decoder_labels[42] = ""

# Special MMS tokens -> single-character placeholders
# so pyctcdecode does not interpret them as multi-character units
decoder_labels[40] = "¤"   # [UNK]
decoder_labels[43] = "«"   # <s>
decoder_labels[44] = "»"   # </s>

# ID 45 remains the literal word space
assert decoder_labels[45] == " "

print("Decoder labels:", len(decoder_labels))

for i, label in enumerate(decoder_labels):
    print(i, repr(label))

Decoder labels: 46
0 '!'
1 ','
2 '-'
3 '.'
4 '?'
5 'a'
6 'b'
7 'd'
8 'e'
9 'f'
10 'g'
11 'h'
12 'i'
13 'j'
14 'k'
15 'l'
16 'm'
17 'n'
18 'p'
19 'q'
20 'r'
21 's'
22 't'
23 'u'
24 'w'
25 'x'
26 'y'
27 'z'
28 'ç'
29 'ā'
30 'č'
31 'ŋ'
32 'š'
33 'ə'
34 'ɛ'
35 'ɣ'
36 'ḍ'
37 'ḥ'
38 'ṣ'
39 'ṭ'
40 '¤'
41 'ẓ'
42 ''
43 '«'
44 '»'
45 ' '


In [ ]:
# ============================================================
# CELL 11: Verify decoder/model dimensional compatibility
# ============================================================

print("Model output classes:", model.config.vocab_size)
print("Decoder labels:", len(decoder_labels))

assert len(decoder_labels) == model.config.vocab_size

print("✓ Decoder labels match MMS logits dimension.")

Model output classes: 46
Decoder labels: 46
✓ Decoder labels match MMS logits dimension.


In [ ]:
# ============================================================
# FIX: Install KenLM properly in current Colab runtime
# ============================================================

!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    cmake \
    libboost-system-dev \
    libboost-thread-dev \
    libboost-program-options-dev \
    libboost-test-dev \
    libeigen3-dev \
    zlib1g-dev \
    libbz2-dev \
    liblzma-dev

# Install pyctcdecode separately
!pip install -q pyctcdecode

# Fresh KenLM source
!rm -rf /content/kenlm
!git clone -q https://github.com/kpu/kenlm.git /content/kenlm

# Build command-line tools
!mkdir -p /content/kenlm/build
!cd /content/kenlm/build && cmake .. > /dev/null
!cd /content/kenlm/build && make -j2 > /dev/null

# Install the Python bindings from the same source tree
!CMAKE_BUILD_PARALLEL_LEVEL=2 pip install -q /content/kenlm

print("KenLM installation attempt finished.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-atomic1.74.0:amd64.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../00-libboost-atomic1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-atomic1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-atomic1.74-dev:amd64.
Preparing to unpack .../01-libboost-atomic1.74-dev_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-atomic1.74-dev:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-chrono1.74.0:amd64.
Preparing to unpack .../02-libboost-chrono1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-chrono1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-chrono1.74-dev:amd64.
Preparing to unpack .../03-libb

In [ ]:
# ============================================================
# VERIFY
# ============================================================

import kenlm
import pyctcdecode
from pathlib import Path

LMPLZ = "/content/kenlm/build/bin/lmplz"
BUILD_BINARY = "/content/kenlm/build/bin/build_binary"

print("KenLM Python: OK")
print("pyctcdecode: OK")
print("lmplz:", Path(LMPLZ).exists())
print("build_binary:", Path(BUILD_BINARY).exists())

KenLM Python: OK
pyctcdecode: OK
lmplz: True
build_binary: True


In [ ]:
# ============================================================
# FIX: Remove shadowing /content/kenlm namespace
# ============================================================

import os
import sys
import shutil

# Remove cloned source folder that is shadowing the real package
if os.path.exists("/content/kenlm"):
    shutil.rmtree("/content/kenlm")
    print("Removed /content/kenlm")

# Remove cached bad import
sys.modules.pop("kenlm", None)

# Make sure current directory is not pointing at deleted kenlm source
print("Current working directory:", os.getcwd())
print("Python:", sys.version)

Removed /content/kenlm
Current working directory: /content
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [ ]:
# ============================================================
# REINSTALL clean Python binding
# ============================================================

!pip install -q --force-reinstall kenlm-ex

print("Reinstall finished.")

Reinstall finished.


In [ ]:
# ============================================================
# VERIFY KENLM AFTER RESTART
# ============================================================

import kenlm

print("KenLM file:", getattr(kenlm, "__file__", None))
print("Has Model:", hasattr(kenlm, "Model"))
print("Available attributes:")
print([x for x in dir(kenlm) if not x.startswith("_")][:30])

KenLM file: /usr/local/lib/python3.13/dist-packages/kenlm.cpython-313-x86_64-linux-gnu.so
Has Model: True
Available attributes:
['ARPALoadComplain', 'Config', 'FullScoreReturn', 'LanguageModel', 'LoadMethod', 'Model', 'State', 'os']


In [ ]:
# ============================================================
# CELL 12: Install KenLM and pyctcdecode
# ============================================================

!pip install -q pyctcdecode kenlm

print("Installation finished.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for kenlm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for kenlm
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (kenlm)
Installation finished.


In [ ]:
# ============================================================
# CELL 13: Check KenLM installation
# ============================================================

import kenlm
import pyctcdecode

print("KenLM Python package: OK")
print("pyctcdecode: OK")

!which lmplz
!which build_binary

KenLM Python package: OK
pyctcdecode: OK


In [ ]:
# ============================================================
# CELL 14: Build KenLM command-line tools
# ============================================================

!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    cmake \
    libboost-system-dev \
    libboost-thread-dev \
    libboost-program-options-dev \
    libboost-test-dev \
    libeigen3-dev \
    zlib1g-dev \
    libbz2-dev \
    liblzma-dev

!rm -rf /content/kenlm
!git clone -q https://github.com/kpu/kenlm.git /content/kenlm

!mkdir -p /content/kenlm/build
!cd /content/kenlm/build && cmake .. > /dev/null
!cd /content/kenlm/build && make -j2 > /dev/null

print("KenLM build finished.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

KenLM build finished.


In [ ]:
# ============================================================
# CELL 15: Verify KenLM binaries
# ============================================================

LMPLZ = "/content/kenlm/build/bin/lmplz"
BUILD_BINARY = "/content/kenlm/build/bin/build_binary"

from pathlib import Path

print("lmplz:", Path(LMPLZ).exists())
print("build_binary:", Path(BUILD_BINARY).exists())

lmplz: True
build_binary: True


In [ ]:
# ============================================================
# CELL 16: Train Tarifit 3-gram language model
# ============================================================

ARPA_PATH = (
    LM_DIR
    / "tarifit_train_3gram.arpa"
)

!{LMPLZ} \
    -o 3 \
    --text "{LM_TEXT_PATH}" \
    --arpa "{ARPA_PATH}" \
    --discount_fallback

print("ARPA model saved to:")
print(ARPA_PATH)

=== 1/5 Counting and sorting n-grams ===
Reading /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/tarifit_lm/train_mms_orthography.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 36185 types 3980
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:47760 2:3785953792 3:7098663424
Statistics:
1 3980 D1=0.630405 D2=1.24464 D3+=1.44751
2 18932 D1=0.791151 D2=1.13801 D3+=1.47872
3 28544 D1=0.819205 D2=1.48747 D3+=1.42527
Memory estimate for binary LM:
type      kB
probing 1046 assuming -p 1.5
probing 1173 assuming -r models -p 1.5
trie     451 without quantization
trie     265 assuming -q 8 -b 8 quantization 
trie     433 assuming -a 22 array pointer compression
trie     248 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initia

In [ ]:
# ============================================================
# CELL 17: Convert Tarifit LM to binary KenLM format
# ============================================================

BINARY_LM_PATH = (
    LM_DIR
    / "tarifit_train_3gram.binary"
)

!{BUILD_BINARY} \
    "{ARPA_PATH}" \
    "{BINARY_LM_PATH}"

print("Binary LM saved to:")
print(BINARY_LM_PATH)

Reading /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/tarifit_lm/tarifit_train_3gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
SUCCESS
Binary LM saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/tarifit_lm/tarifit_train_3gram.binary


In [ ]:
# ============================================================
# VERIFY KENLM
# ============================================================

import kenlm
import pyctcdecode

print("KenLM module:", kenlm.__file__)
print("Has Model:", hasattr(kenlm, "Model"))

lm = kenlm.Model(str(BINARY_LM_PATH))

print("KenLM model loaded successfully.")
print(
    "Test score:",
    lm.score("di lmeɣrib")
)

KenLM module: /usr/local/lib/python3.13/dist-packages/kenlm.cpython-313-x86_64-linux-gnu.so
Has Model: True
KenLM model loaded successfully.
Test score: -8.221884727478027


In [ ]:
# Remove the source folder from the current working import path
import os
import sys
import importlib

if "/content" in sys.path:
    sys.path.remove("/content")

# Remove already-imported wrong module
sys.modules.pop("kenlm", None)

# Re-import
import kenlm

print("kenlm file:", getattr(kenlm, "__file__", None))
print("Has Model:", hasattr(kenlm, "Model"))

kenlm file: /usr/local/lib/python3.13/dist-packages/kenlm.cpython-313-x86_64-linux-gnu.so
Has Model: True


In [ ]:
# ============================================================
# CELL 18: Test Tarifit KenLM model
# ============================================================

import kenlm

lm = kenlm.Model(
    str(BINARY_LM_PATH)
)

test_sentences = [
    "di lmeɣrib",
    "nešin mamšira",
    "rexbar asbḥan n yasuɛ lmasiḥ",
]

for sentence in test_sentences:
    print(
        sentence,
        "-> score:",
        lm.score(sentence)
    )

di lmeɣrib -> score: -8.221884727478027
nešin mamšira -> score: -10.501160621643066
rexbar asbḥan n yasuɛ lmasiḥ -> score: -5.50575065612793


In [ ]:
# ============================================================
# CELL 18B: Import pyctcdecode
# ============================================================

from pyctcdecode import build_ctcdecoder

print("build_ctcdecoder imported successfully")

build_ctcdecoder imported successfully


In [ ]:
# ============================================================
# CELL 19: Build plain CTC beam-search decoder
# ============================================================

decoder_beam = build_ctcdecoder(
    labels=decoder_labels
)

print("Plain beam-search decoder ready.")

Plain beam-search decoder ready.


In [ ]:
# ============================================================
# CELL 20A: Build known Tarifit unigrams for KenLM decoding
# ============================================================

lm_unigrams = sorted({
    word
    for sentence in lm_texts
    for word in sentence.split()
})

print("Number of LM unigrams:", len(lm_unigrams))
print("Examples:", lm_unigrams[:30])

Number of LM unigrams: 3977
Examples: ['a', 'abarrani', 'abarreḥ', 'abarru', 'abaršan', 'abbas', 'abdur', 'abeḥrur', 'abihud', 'abiya', 'abrid', 'abriɣ', 'abuhari', 'aburay', 'abyatar', 'ad', 'adaf', 'adarɣar', 'adas', 'add', 'adef', 'adet', 'adfem', 'adfen', 'adfent', 'adfer', 'adhan', 'adimreš', 'adlis', 'adwar']


In [ ]:
# ============================================================
# CELL 20: Build CTC beam search + Tarifit KenLM
# ============================================================

from pyctcdecode import build_ctcdecoder

decoder_lm = build_ctcdecoder(
    labels=decoder_labels,
    kenlm_model_path=str(BINARY_LM_PATH),
    unigrams=lm_unigrams,

    # Initial values — we'll tune later
    alpha=0.5,
    beta=1.0,
)

print("Tarifit KenLM decoder ready.")

Tarifit KenLM decoder ready.


In [ ]:
# ============================================================
# CELL 21: Normalize MMS decoder output
# ============================================================

import re
import unicodedata

def normalize_mms_output(text):
    text = unicodedata.normalize("NFC", str(text)).lower()

    # Remove decoder placeholders
    text = text.replace("¤", " ")
    text = text.replace("«", " ")
    text = text.replace("»", " ")

    # MMS -> Corpus V1.1 orthography
    text = text.replace("ə", "e")
    text = text.replace("š", "c")

    # Remove punctuation
    text = re.sub(r"[!?,.\-]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# ============================================================
# CELL 22: Load clean validation metadata
# ============================================================

import pandas as pd

validation_df = pd.read_csv(VALIDATION_CSV)

BAD_VALIDATION_IDS = {
    "REC138_SEG0025",
    "REC138_SEG0033",
    "REC138_SEG0041",
    "REC138_SEG0042",
    "REC138_SEG0045",
}

validation_clean_df = (
    validation_df[
        ~validation_df["segment_id"].isin(BAD_VALIDATION_IDS)
    ]
    .reset_index(drop=True)
)

print("Original validation:", len(validation_df))
print("Clean validation:", len(validation_clean_df))

Original validation: 133
Clean validation: 128


In [ ]:
# ============================================================
# CELL 23: Prepare MMS zero-shot model on GPU
# ============================================================

import torch

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(DEVICE)
model.eval()

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 24: Generate MMS logits for clean validation
# ============================================================

import numpy as np
import soundfile as sf
from tqdm.auto import tqdm

LOGITS_DIR = (
    PROJECT_ROOT
    / "results"
    / "mms_kenlm"
    / "validation_logits"
)

LOGITS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

logit_manifest = []

for _, row in tqdm(
    validation_clean_df.iterrows(),
    total=len(validation_clean_df)
):

    segment_id = row["segment_id"]

    audio_file = (
        PROJECT_ROOT
        / row["audio_path"]
    )

    audio, sampling_rate = sf.read(
        audio_file
    )

    inputs = processor(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    input_values = (
        inputs["input_values"]
        .to(DEVICE)
    )

    attention_mask = inputs.get(
        "attention_mask"
    )

    if attention_mask is not None:
        attention_mask = (
            attention_mask.to(DEVICE)
        )

    with torch.no_grad():

        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask
        )

    # Save logits as float16 to reduce disk usage
    logits = (
        outputs.logits[0]
        .detach()
        .cpu()
        .numpy()
        .astype(np.float16)
    )

    logits_path = (
        LOGITS_DIR
        / f"{segment_id}.npy"
    )

    np.save(
        logits_path,
        logits
    )

    logit_manifest.append({
        "segment_id": segment_id,
        "speaker_group_id": row["speaker_group_id"],
        "reference": row["transcription"],
        "logits_path": str(logits_path),
        "num_frames": logits.shape[0],
        "num_classes": logits.shape[1],
    })

print("\nGenerated logits:", len(logit_manifest))

  0%|          | 0/128 [00:00<?, ?it/s]


Generated logits: 128


In [ ]:
# ============================================================
# CELL 25: Save logits manifest
# ============================================================

manifest_df = pd.DataFrame(
    logit_manifest
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "results"
    / "mms_kenlm"
    / "validation_logits_manifest.csv"
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved manifest:")
print(MANIFEST_PATH)

print("\nSegments:", len(manifest_df))
print(
    "Classes:",
    manifest_df["num_classes"].unique()
)

Saved manifest:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_kenlm/validation_logits_manifest.csv

Segments: 128
Classes: [46]


In [ ]:
# ============================================================
# CELL 26: Greedy decoding from saved logits
# ============================================================

greedy_results = []

for _, row in manifest_df.iterrows():

    logits = np.load(
        row["logits_path"]
    ).astype(np.float32)

    predicted_ids = np.argmax(
        logits,
        axis=-1
    )

    prediction_raw = (
        processor.decode(
            predicted_ids
        )
    )

    prediction_normalized = (
        normalize_mms_output(
            prediction_raw
        )
    )

    greedy_results.append({
        "segment_id": row["segment_id"],
        "reference": row["reference"],
        "prediction_raw": prediction_raw,
        "prediction_normalized": prediction_normalized,
    })

print("Decoded:", len(greedy_results))

Decoded: 128


In [ ]:
# ============================================================
# INSTALL jiwer
# ============================================================

!pip install -q jiwer

print("jiwer installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.9 MB/s eta 0:00:00
jiwer installed.


In [ ]:
# ============================================================
# CELL 27: Verify saved-logit greedy baseline
# ============================================================

from jiwer import wer, cer

refs = [
    x["reference"]
    for x in greedy_results
]

raw_hyps = [
    x["prediction_raw"]
    for x in greedy_results
]

norm_hyps = [
    x["prediction_normalized"]
    for x in greedy_results
]

print("GREEDY — RAW")
print(
    "WER:",
    round(wer(refs, raw_hyps) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(refs, raw_hyps) * 100, 2),
    "%"
)

print("\nGREEDY — NORMALIZED")
print(
    "WER:",
    round(wer(refs, norm_hyps) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(refs, norm_hyps) * 100, 2),
    "%"
)

GREEDY — RAW
WER: 93.11 %
CER: 49.26 %

GREEDY — NORMALIZED
WER: 78.92 %
CER: 41.01 %


In [ ]:
# ============================================================
# CELL 28: Plain CTC beam search — quick 10-sample test
# ============================================================

beam_test_results = []

for _, row in manifest_df.head(10).iterrows():

    logits = np.load(
        row["logits_path"]
    ).astype(np.float32)

    prediction_raw = decoder_beam.decode(
        logits,
        beam_width=50
    )

    prediction_normalized = normalize_mms_output(
        prediction_raw
    )

    beam_test_results.append({
        "segment_id": row["segment_id"],
        "reference": row["reference"],
        "prediction_raw": prediction_raw,
        "prediction_normalized": prediction_normalized,
    })

print("Decoded:", len(beam_test_results))

for x in beam_test_results[:5]:
    print("\n", x["segment_id"])
    print("REF :", x["reference"])
    print("BEAM:", x["prediction_normalized"])

Decoded: 10

 REC090_SEG0010
REF : ssalamuɛlikum necc meryem
BEAM: ttbmbnwɣmjlwn pɛe nɛszɛn

 REC090_SEG0011
REF : aqay ruxxa tnayn uɛecrin sana di hulanda
BEAM: brb z swyyb upɛp wɣɛesjp tbpb e j iwmbpeb

 REC090_SEG0012
REF : mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca
BEAM: nɛseɛy bl njtɛpkkjsbp wtjḍ e çj mnɛḍsjd wnjsb rrb zej mnɛḍsjd jsɛrrb ḍbt be sɛṣ ḍbs wswqb be hhɛy be hɛynbeb

 REC090_SEG0013
REF : umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu
BEAM: wnj xtjḍ eeb wgj y nbpbzɛppj xb kj eb njsjsbḍbsj eeij mɣrɛm jpw

 REC090_SEG0014
REF : a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uffix dda
BEAM: pɛenbnej sb kj y eij mnɛḍsjd xbkj nbpbzɛppj wgj y eeb


In [ ]:
# ============================================================
# CELL 29: Quick greedy vs beam comparison
# ============================================================

test_ids = {
    x["segment_id"]
    for x in beam_test_results
}

greedy_10 = [
    x for x in greedy_results
    if x["segment_id"] in test_ids
]

refs_10 = [
    x["reference"]
    for x in beam_test_results
]

beam_hyps_10 = [
    x["prediction_normalized"]
    for x in beam_test_results
]

greedy_hyps_10 = [
    x["prediction_normalized"]
    for x in greedy_10
]

print("GREEDY — 10 samples")
print("WER:", round(wer(refs_10, greedy_hyps_10) * 100, 2))
print("CER:", round(cer(refs_10, greedy_hyps_10) * 100, 2))

print("\nPLAIN BEAM — 10 samples")
print("WER:", round(wer(refs_10, beam_hyps_10) * 100, 2))
print("CER:", round(cer(refs_10, beam_hyps_10) * 100, 2))

GREEDY — 10 samples
WER: 65.41
CER: 15.83

PLAIN BEAM — 10 samples
WER: 101.26
CER: 83.44


In [ ]:
# ============================================================
# CELL 30: Plain CTC beam search — full clean validation
# ============================================================

from tqdm.auto import tqdm

beam_results = []

for _, row in tqdm(
    manifest_df.iterrows(),
    total=len(manifest_df)
):

    logits = np.load(
        row["logits_path"]
    ).astype(np.float32)

    prediction_raw = decoder_beam.decode(
        logits,
        beam_width=50
    )

    prediction_normalized = normalize_mms_output(
        prediction_raw
    )

    beam_results.append({
        "segment_id": row["segment_id"],
        "speaker_group_id": row["speaker_group_id"],
        "reference": row["reference"],
        "prediction_raw": prediction_raw,
        "prediction_normalized": prediction_normalized,
    })

print("Decoded:", len(beam_results))

  0%|          | 0/128 [00:00<?, ?it/s]

Decoded: 128


In [ ]:
# ============================================================
# CELL 31: Evaluate plain beam search
# ============================================================

beam_refs = [
    x["reference"]
    for x in beam_results
]

beam_raw = [
    x["prediction_raw"]
    for x in beam_results
]

beam_norm = [
    x["prediction_normalized"]
    for x in beam_results
]

print("PLAIN BEAM — RAW")
print(
    "WER:",
    round(wer(beam_refs, beam_raw) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(beam_refs, beam_raw) * 100, 2),
    "%"
)

print("\nPLAIN BEAM — NORMALIZED")
print(
    "WER:",
    round(wer(beam_refs, beam_norm) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(beam_refs, beam_norm) * 100, 2),
    "%"
)

PLAIN BEAM — RAW
WER: 110.43 %
CER: 97.04 %

PLAIN BEAM — NORMALIZED
WER: 111.39 %
CER: 95.21 %


In [ ]:
# ============================================================
# CELL 32: Inspect MMS vocabulary and CTC blank configuration
# ============================================================

vocab = processor.tokenizer.get_vocab()

id_to_token = {
    idx: token
    for token, idx in vocab.items()
}

print("Model vocab size:", model.config.vocab_size)
print("Tokenizer vocab size:", len(vocab))
print("Model pad_token_id:", model.config.pad_token_id)
print("Tokenizer pad_token_id:", processor.tokenizer.pad_token_id)

print("\n--- TOKEN IDS ---")
for i in range(model.config.vocab_size):
    print(
        f"{i:2d}",
        repr(id_to_token.get(i, "<MISSING>"))
    )

Model vocab size: 46
Tokenizer vocab size: 46
Model pad_token_id: 42
Tokenizer pad_token_id: 42

--- TOKEN IDS ---
 0 '<MISSING>'
 1 '!'
 2 ','
 3 '-'
 4 '.'
 5 '?'
 6 'a'
 7 'b'
 8 'd'
 9 'e'
10 'f'
11 'g'
12 'h'
13 'i'
14 'j'
15 'k'
16 'l'
17 'm'
18 'n'
19 'p'
20 'q'
21 'r'
22 's'
23 't'
24 'u'
25 'w'
26 'x'
27 'y'
28 'z'
29 'ç'
30 'ā'
31 'č'
32 'ŋ'
33 'š'
34 'ə'
35 'ɛ'
36 'ɣ'
37 'ḍ'
38 'ḥ'
39 'ṣ'
40 'ṭ'
41 'ẓ'
42 '[PAD]'
43 '<s>'
44 '</s>'
45 ' '


In [ ]:
# ============================================================
# CELL 32B: Diagnose MMS tokenizer ID mapping
# ============================================================

from collections import defaultdict

vocab = processor.tokenizer.get_vocab()

# Show every token -> ID mapping sorted by ID
print("=== RAW get_vocab() ===")
for token, idx in sorted(vocab.items(), key=lambda x: (x[1], x[0])):
    print(f"{idx:2d}  {repr(token)}")

# Find IDs assigned to more than one token
id_groups = defaultdict(list)

for token, idx in vocab.items():
    id_groups[idx].append(token)

print("\n=== DUPLICATE IDs ===")
for idx, tokens in sorted(id_groups.items()):
    if len(tokens) > 1:
        print(idx, tokens)

print("\n=== MISSING IDs ===")
used_ids = set(vocab.values())

for i in range(model.config.vocab_size):
    if i not in used_ids:
        print("Missing:", i)

print("\n=== convert_ids_to_tokens() ===")
for i in range(model.config.vocab_size):
    token = processor.tokenizer.convert_ids_to_tokens(i)
    print(f"{i:2d} -> {repr(token)}")

=== RAW get_vocab() ===
 1  '!'
 2  ','
 3  '-'
 4  '.'
 5  '?'
 6  'a'
 7  'b'
 8  'd'
 9  'e'
10  'f'
11  'g'
12  'h'
13  'i'
14  'j'
15  'k'
16  'l'
17  'm'
18  'n'
19  'p'
20  'q'
21  'r'
22  's'
23  't'
24  'u'
25  'w'
26  'x'
27  'y'
28  'z'
29  'ç'
30  'ā'
31  'č'
32  'ŋ'
33  'š'
34  'ə'
35  'ɛ'
36  'ɣ'
37  'ḍ'
38  'ḥ'
39  'ṣ'
40  'ṭ'
41  '[UNK]'
41  'ẓ'
42  '[PAD]'
43  '<s>'
44  '</s>'
45  ' '

=== DUPLICATE IDs ===
41 ['[UNK]', 'ẓ']

=== MISSING IDs ===
Missing: 0

=== convert_ids_to_tokens() ===
 0 -> '[UNK]'
 1 -> '!'
 2 -> ','
 3 -> '-'
 4 -> '.'
 5 -> '?'
 6 -> 'a'
 7 -> 'b'
 8 -> 'd'
 9 -> 'e'
10 -> 'f'
11 -> 'g'
12 -> 'h'
13 -> 'i'
14 -> 'j'
15 -> 'k'
16 -> 'l'
17 -> 'm'
18 -> 'n'
19 -> 'p'
20 -> 'q'
21 -> 'r'
22 -> 's'
23 -> 't'
24 -> 'u'
25 -> 'w'
26 -> 'x'
27 -> 'y'
28 -> 'z'
29 -> 'ç'
30 -> 'ā'
31 -> 'č'
32 -> 'ŋ'
33 -> 'š'
34 -> 'ə'
35 -> 'ɛ'
36 -> 'ɣ'
37 -> 'ḍ'
38 -> 'ḥ'
39 -> 'ṣ'
40 -> 'ṭ'
41 -> 'ẓ'
42 -> '[PAD]'
43 -> '<s>'
44 -> '</s>'
45 -> ' '


In [ ]:
# ============================================================
# CELL 32C: Probe processor decoding for every output ID
# ============================================================

for i in range(model.config.vocab_size):

    token = processor.tokenizer.convert_ids_to_tokens(i)

    decoded_keep_special = processor.decode(
        [i],
        skip_special_tokens=False
    )

    decoded_skip_special = processor.decode(
        [i],
        skip_special_tokens=True
    )

    print(
        f"{i:2d} | "
        f"token={repr(token):12s} | "
        f"decode={repr(decoded_keep_special):12s} | "
        f"skip={repr(decoded_skip_special)}"
    )

 0 | token='[UNK]'      | decode='[UNK]'      | skip=''
 1 | token='!'          | decode='!'          | skip='!'
 2 | token=','          | decode=','          | skip=','
 3 | token='-'          | decode='-'          | skip='-'
 4 | token='.'          | decode='.'          | skip='.'
 5 | token='?'          | decode='?'          | skip='?'
 6 | token='a'          | decode='a'          | skip='a'
 7 | token='b'          | decode='b'          | skip='b'
 8 | token='d'          | decode='d'          | skip='d'
 9 | token='e'          | decode='e'          | skip='e'
10 | token='f'          | decode='f'          | skip='f'
11 | token='g'          | decode='g'          | skip='g'
12 | token='h'          | decode='h'          | skip='h'
13 | token='i'          | decode='i'          | skip='i'
14 | token='j'          | decode='j'          | skip='j'
15 | token='k'          | decode='k'          | skip='k'
16 | token='l'          | decode='l'          | skip='l'
17 | token='m'          | decode

In [ ]:
# ============================================================
# CELL 32D: Build CORRECT decoder labels from model output IDs
# ============================================================

decoder_labels = []

for i in range(model.config.vocab_size):

    token = processor.tokenizer.convert_ids_to_tokens(i)

    # CTC blank
    if i == model.config.pad_token_id:      # ID 42
        label = ""

    # Unknown output class — single-character placeholder
    elif i == 0:
        label = "¤"

    # BOS / EOS — single-character placeholders
    elif i == 43:
        label = "«"

    elif i == 44:
        label = "»"

    else:
        label = token

    decoder_labels.append(label)


print("Number of labels:", len(decoder_labels))

print("\n=== CORRECT DECODER LABELS ===")

for i, label in enumerate(decoder_labels):
    original = processor.tokenizer.convert_ids_to_tokens(i)

    print(
        f"{i:2d} | "
        f"MMS={repr(original):10s} | "
        f"decoder={repr(label)}"
    )

Number of labels: 46

=== CORRECT DECODER LABELS ===
 0 | MMS='[UNK]'    | decoder='¤'
 1 | MMS='!'        | decoder='!'
 2 | MMS=','        | decoder=','
 3 | MMS='-'        | decoder='-'
 4 | MMS='.'        | decoder='.'
 5 | MMS='?'        | decoder='?'
 6 | MMS='a'        | decoder='a'
 7 | MMS='b'        | decoder='b'
 8 | MMS='d'        | decoder='d'
 9 | MMS='e'        | decoder='e'
10 | MMS='f'        | decoder='f'
11 | MMS='g'        | decoder='g'
12 | MMS='h'        | decoder='h'
13 | MMS='i'        | decoder='i'
14 | MMS='j'        | decoder='j'
15 | MMS='k'        | decoder='k'
16 | MMS='l'        | decoder='l'
17 | MMS='m'        | decoder='m'
18 | MMS='n'        | decoder='n'
19 | MMS='p'        | decoder='p'
20 | MMS='q'        | decoder='q'
21 | MMS='r'        | decoder='r'
22 | MMS='s'        | decoder='s'
23 | MMS='t'        | decoder='t'
24 | MMS='u'        | decoder='u'
25 | MMS='w'        | decoder='w'
26 | MMS='x'        | decoder='x'
27 | MMS='y'        | decoder

In [ ]:
# ============================================================
# CELL 32E: Rebuild plain CTC decoder
# ============================================================

from pyctcdecode import build_ctcdecoder

decoder_beam = build_ctcdecoder(
    labels=decoder_labels
)

print("Corrected plain CTC decoder built.")

Corrected plain CTC decoder built.


In [ ]:
# ============================================================
# CELL 32F: One-segment decoder sanity check
# ============================================================

row = manifest_df.iloc[0]

logits = np.load(
    row["logits_path"]
).astype(np.float32)

argmax_ids = np.argmax(
    logits,
    axis=-1
)

greedy_processor = processor.decode(
    argmax_ids
)

beam_1 = decoder_beam.decode(
    logits,
    beam_width=1
)

beam_10 = decoder_beam.decode(
    logits,
    beam_width=10
)

beam_50 = decoder_beam.decode(
    logits,
    beam_width=50
)

print("SEGMENT:", row["segment_id"])

print("\nREFERENCE:")
print(row["reference"])

print("\nPROCESSOR GREEDY:")
print(greedy_processor)

print("\nBEAM WIDTH 1:")
print(beam_1)

print("\nBEAM WIDTH 10:")
print(beam_10)

print("\nBEAM WIDTH 50:")
print(beam_50)

print("\n--- NORMALIZED ---")

print(
    "GREEDY:",
    normalize_mms_output(greedy_processor)
)

print(
    "BEAM 1:",
    normalize_mms_output(beam_1)
)

print(
    "BEAM 10:",
    normalize_mms_output(beam_10)
)

print(
    "BEAM 50:",
    normalize_mms_output(beam_50)
)

SEGMENT: REC090_SEG0010

REFERENCE:
ssalamuɛlikum necc meryem

PROCESSOR GREEDY:
ssalamuɛlikum, nəš məryəm.

BEAM WIDTH 1:
ssalamuɛlikum, nəš məryəm.

BEAM WIDTH 10:
ssalam uɛlikum, nəš məryəm.

BEAM WIDTH 50:
ssalamuɛlikum, nəš məryəm.

--- NORMALIZED ---
GREEDY: ssalamuɛlikum nec meryem
BEAM 1: ssalamuɛlikum nec meryem
BEAM 10: ssalam uɛlikum nec meryem
BEAM 50: ssalamuɛlikum nec meryem


In [ ]:
# ============================================================
# CELL 32G: Corrected plain beam — 10 samples
# ============================================================

corrected_beam_10 = []

for _, row in manifest_df.head(10).iterrows():

    logits = np.load(
        row["logits_path"]
    ).astype(np.float32)

    prediction_raw = decoder_beam.decode(
        logits,
        beam_width=50
    )

    prediction_norm = normalize_mms_output(
        prediction_raw
    )

    corrected_beam_10.append(
        prediction_norm
    )


refs_10 = (
    manifest_df.head(10)["reference"]
    .tolist()
)

greedy_hyps_10 = [
    x["prediction_normalized"]
    for x in greedy_results[:10]
]

print("GREEDY — 10 samples")
print(
    "WER:",
    round(
        wer(refs_10, greedy_hyps_10) * 100,
        2
    )
)
print(
    "CER:",
    round(
        cer(refs_10, greedy_hyps_10) * 100,
        2
    )
)

print("\nCORRECTED PLAIN BEAM — 10 samples")
print(
    "WER:",
    round(
        wer(refs_10, corrected_beam_10) * 100,
        2
    )
)
print(
    "CER:",
    round(
        cer(refs_10, corrected_beam_10) * 100,
        2
    )
)

GREEDY — 10 samples
WER: 65.41
CER: 15.83

CORRECTED PLAIN BEAM — 10 samples
WER: 65.41
CER: 15.83


In [ ]:
# ============================================================
# CELL 33: Corrected plain CTC beam — full validation
# ============================================================

from tqdm.auto import tqdm

beam_results = []

for _, row in tqdm(
    manifest_df.iterrows(),
    total=len(manifest_df)
):

    logits = np.load(
        row["logits_path"]
    ).astype(np.float32)

    prediction_raw = decoder_beam.decode(
        logits,
        beam_width=50
    )

    prediction_normalized = normalize_mms_output(
        prediction_raw
    )

    beam_results.append({
        "segment_id": row["segment_id"],
        "speaker_group_id": row["speaker_group_id"],
        "reference": row["reference"],
        "prediction_raw": prediction_raw,
        "prediction_normalized": prediction_normalized,
    })

print("Decoded:", len(beam_results))

  0%|          | 0/128 [00:00<?, ?it/s]

Decoded: 128


In [ ]:
# ============================================================
# CELL 34: Evaluate corrected plain CTC beam
# ============================================================

beam_refs = [
    x["reference"]
    for x in beam_results
]

beam_raw = [
    x["prediction_raw"]
    for x in beam_results
]

beam_norm = [
    x["prediction_normalized"]
    for x in beam_results
]

print("CORRECTED PLAIN BEAM — RAW")
print(
    "WER:",
    round(wer(beam_refs, beam_raw) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(beam_refs, beam_raw) * 100, 2),
    "%"
)

print("\nCORRECTED PLAIN BEAM — NORMALIZED")
print(
    "WER:",
    round(wer(beam_refs, beam_norm) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(beam_refs, beam_norm) * 100, 2),
    "%"
)

CORRECTED PLAIN BEAM — RAW
WER: 94.57 %
CER: 49.56 %

CORRECTED PLAIN BEAM — NORMALIZED
WER: 81.09 %
CER: 41.38 %


Plain CTC beam search did not outperform greedy decoding. Its CER remained close to greedy decoding, but WER increased, suggesting that unconstrained beam search altered lexical/word-boundary decisions without providing useful linguistic guidance.

In [ ]:
# ============================================================
# CELL 35: Verify KenLM and LM files
# ============================================================

import kenlm
from pathlib import Path

print("Has kenlm.Model:", hasattr(kenlm, "Model"))
print("Binary LM exists:", Path(BINARY_LM_PATH).exists())
print("Binary LM path:", BINARY_LM_PATH)

lm_test = kenlm.Model(str(BINARY_LM_PATH))

print("KenLM loaded successfully.")
print(
    "Example score:",
    lm_test.score("di lmeɣrib")
)

Has kenlm.Model: True
Binary LM exists: True
Binary LM path: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/tarifit_lm/tarifit_train_3gram.binary
KenLM loaded successfully.
Example score: -8.221884727478027


In [ ]:
# ============================================================
# CELL 36: Build CTC + KenLM decoder
# ============================================================

from pyctcdecode import build_ctcdecoder

decoder_lm = build_ctcdecoder(
    labels=decoder_labels,
    kenlm_model_path=str(BINARY_LM_PATH),
    unigrams=lm_unigrams,

    # Initial conservative settings
    alpha=0.5,
    beta=1.0,
)

print("CTC + KenLM decoder built successfully.")

CTC + KenLM decoder built successfully.


In [ ]:
# ============================================================
# CELL 37: KenLM beam decoding — first 10 samples
# ============================================================

lm_test_results = []

for _, row in manifest_df.head(10).iterrows():

    logits = np.load(
        row["logits_path"]
    ).astype(np.float32)

    prediction_raw = decoder_lm.decode(
        logits,
        beam_width=50
    )

    prediction_normalized = normalize_mms_output(
        prediction_raw
    )

    lm_test_results.append({
        "segment_id": row["segment_id"],
        "reference": row["reference"],
        "prediction_raw": prediction_raw,
        "prediction_normalized": prediction_normalized,
    })


print("Decoded:", len(lm_test_results))

for x in lm_test_results[:5]:

    print("\n", x["segment_id"])

    print(
        "REF :",
        x["reference"]
    )

    print(
        "LM  :",
        x["prediction_normalized"]
    )

Decoded: 10

 REC090_SEG0010
REF : ssalamuɛlikum necc meryem
LM  : ssalamuɛlikum n meryem

 REC090_SEG0011
REF : aqay ruxxa tnayn uɛecrin sana di hulanda
LM  : aqa ruxatnenuɛcrin sanadihulanda

 REC090_SEG0012
REF : mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca
LM  : mercex ad misenjjiran usiɣ d zi lmeɣrib umi aqa di lmeɣribireqqa as ad rḥar urupaadegex ad maca

 REC090_SEG0013
REF : umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu
LM  : umi usiɣ da ufixmanyenni wa i ca miriraɣariddhi lɛqel inu

 REC090_SEG0014
REF : a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uffix dda
LM  : n maci ra i x di lmeɣrib waji manayenniufi x dda


In [ ]:
# ============================================================
# CELL 38: Greedy vs Beam vs Beam+KenLM — 10 samples
# ============================================================

refs_10 = (
    manifest_df.head(10)["reference"]
    .tolist()
)

greedy_10 = [
    x["prediction_normalized"]
    for x in greedy_results[:10]
]

plain_beam_10 = []

for _, row in manifest_df.head(10).iterrows():

    logits = np.load(
        row["logits_path"]
    ).astype(np.float32)

    pred = decoder_beam.decode(
        logits,
        beam_width=50
    )

    plain_beam_10.append(
        normalize_mms_output(pred)
    )


lm_10 = [
    x["prediction_normalized"]
    for x in lm_test_results
]


print("GREEDY")
print(
    "WER:",
    round(wer(refs_10, greedy_10) * 100, 2)
)
print(
    "CER:",
    round(cer(refs_10, greedy_10) * 100, 2)
)


print("\nPLAIN BEAM")
print(
    "WER:",
    round(wer(refs_10, plain_beam_10) * 100, 2)
)
print(
    "CER:",
    round(cer(refs_10, plain_beam_10) * 100, 2)
)


print("\nBEAM + KENLM")
print(
    "WER:",
    round(wer(refs_10, lm_10) * 100, 2)
)
print(
    "CER:",
    round(cer(refs_10, lm_10) * 100, 2)
)

GREEDY
WER: 65.41
CER: 15.83

PLAIN BEAM
WER: 65.41
CER: 15.83

BEAM + KENLM
WER: 71.07
CER: 22.82


In [ ]:
# ============================================================
# CELL 39: Small KenLM alpha/beta sweep — 10 samples
# ============================================================

from pyctcdecode import build_ctcdecoder
from itertools import product

alpha_values = [0.1, 0.2, 0.3, 0.5]
beta_values = [0.0, 0.5, 1.0]

grid_results = []

refs_10 = (
    manifest_df.head(10)["reference"]
    .tolist()
)

for alpha, beta in product(alpha_values, beta_values):

    decoder_tmp = build_ctcdecoder(
        labels=decoder_labels,
        kenlm_model_path=str(BINARY_LM_PATH),
        unigrams=lm_unigrams,
        alpha=alpha,
        beta=beta,
    )

    predictions = []

    for _, row in manifest_df.head(10).iterrows():

        logits = np.load(
            row["logits_path"]
        ).astype(np.float32)

        pred = decoder_tmp.decode(
            logits,
            beam_width=50
        )

        predictions.append(
            normalize_mms_output(pred)
        )

    score_wer = wer(
        refs_10,
        predictions
    ) * 100

    score_cer = cer(
        refs_10,
        predictions
    ) * 100

    grid_results.append({
        "alpha": alpha,
        "beta": beta,
        "wer": score_wer,
        "cer": score_cer,
    })

    print(
        f"alpha={alpha:.1f}, beta={beta:.1f} "
        f"-> WER={score_wer:.2f}, CER={score_cer:.2f}"
    )

alpha=0.1, beta=0.0 -> WER=69.81, CER=19.88
alpha=0.1, beta=0.5 -> WER=75.47, CER=20.86
alpha=0.1, beta=1.0 -> WER=79.25, CER=20.98
alpha=0.2, beta=0.0 -> WER=69.18, CER=21.35
alpha=0.2, beta=0.5 -> WER=70.44, CER=21.72
alpha=0.2, beta=1.0 -> WER=74.84, CER=21.23
alpha=0.3, beta=0.0 -> WER=72.33, CER=22.94
alpha=0.3, beta=0.5 -> WER=72.33, CER=22.45
alpha=0.3, beta=1.0 -> WER=73.58, CER=22.45
alpha=0.5, beta=0.0 -> WER=72.33, CER=21.96
alpha=0.5, beta=0.5 -> WER=70.44, CER=22.09
alpha=0.5, beta=1.0 -> WER=71.07, CER=22.82


In [ ]:
# ============================================================
# CELL 40: Rank KenLM settings
# ============================================================

grid_df = pd.DataFrame(
    grid_results
).sort_values(
    ["wer", "cer"]
).reset_index(drop=True)

display(grid_df)

print("\nBest setting:")
print(grid_df.iloc[0])

,alpha,beta,wer,cer
0,0.2,0.0,69.182390,21.349693
1,0.1,0.0,69.811321,19.877301
2,0.2,0.5,70.440252,21.717791
3,0.5,0.5,70.440252,22.085890
4,0.5,1.0,71.069182,22.822086
5,0.5,0.0,72.327044,21.963190
6,0.3,0.5,72.327044,22.453988
7,0.3,0.0,72.327044,22.944785
8,0.3,1.0,73.584906,22.453988
9,0.2,1.0,74.842767,21.226994



Best setting:
alpha     0.200000
beta      0.000000
wer      69.182390
cer      21.349693
Name: 0, dtype: float64


In [ ]:
# ============================================================
# CELL 41: KenLM shortlist — full clean validation
# ============================================================

from pyctcdecode import build_ctcdecoder
from tqdm.auto import tqdm

full_lm_configs = [
    {"alpha": 0.05, "beta": 0.0},
    {"alpha": 0.10, "beta": 0.0},
    {"alpha": 0.20, "beta": 0.0},
]

full_lm_results = []

refs_full = manifest_df["reference"].tolist()

for config in full_lm_configs:

    alpha = config["alpha"]
    beta = config["beta"]

    print(
        f"\nDecoding full validation: "
        f"alpha={alpha}, beta={beta}"
    )

    decoder_tmp = build_ctcdecoder(
        labels=decoder_labels,
        kenlm_model_path=str(BINARY_LM_PATH),
        unigrams=lm_unigrams,
        alpha=alpha,
        beta=beta,
    )

    predictions = []

    for _, row in tqdm(
        manifest_df.iterrows(),
        total=len(manifest_df)
    ):

        logits = np.load(
            row["logits_path"]
        ).astype(np.float32)

        pred_raw = decoder_tmp.decode(
            logits,
            beam_width=50
        )

        pred_norm = normalize_mms_output(
            pred_raw
        )

        predictions.append(pred_norm)

    score_wer = wer(
        refs_full,
        predictions
    ) * 100

    score_cer = cer(
        refs_full,
        predictions
    ) * 100

    full_lm_results.append({
        "alpha": alpha,
        "beta": beta,
        "wer": score_wer,
        "cer": score_cer,
        "predictions": predictions,
    })

    print(
        f"WER={score_wer:.2f}% | "
        f"CER={score_cer:.2f}%"
    )


Decoding full validation: alpha=0.05, beta=0.0


  0%|          | 0/128 [00:00<?, ?it/s]

WER=90.98% | CER=42.42%

Decoding full validation: alpha=0.1, beta=0.0


  0%|          | 0/128 [00:00<?, ?it/s]

WER=87.65% | CER=42.27%

Decoding full validation: alpha=0.2, beta=0.0


  0%|          | 0/128 [00:00<?, ?it/s]

WER=84.77% | CER=42.19%


In [ ]:
# ============================================================
# CELL 42: Final LM1 decoder comparison
# ============================================================

summary_rows = [
    {
        "decoder": "Greedy",
        "alpha": None,
        "beta": None,
        "wer": 78.92,
        "cer": 41.01,
    },
    {
        "decoder": "Plain beam",
        "alpha": 0.0,
        "beta": 0.0,
        "wer": 81.09,
        "cer": 41.38,
    },
]

for result in full_lm_results:

    summary_rows.append({
        "decoder": "Beam + KenLM",
        "alpha": result["alpha"],
        "beta": result["beta"],
        "wer": result["wer"],
        "cer": result["cer"],
    })

summary_df = pd.DataFrame(summary_rows)

display(
    summary_df.sort_values(
        ["wer", "cer"]
    ).reset_index(drop=True)
)

,decoder,alpha,beta,wer,cer
0,Greedy,NaN,NaN,78.920000,41.010000
1,Plain beam,0.00,0.0,81.090000,41.380000
2,Beam + KenLM,0.20,0.0,84.766277,42.185607
3,Beam + KenLM,0.10,0.0,87.646077,42.274453
4,Beam + KenLM,0.05,0.0,90.984975,42.419837


In [ ]:
# ============================================================
# CELL 43: Save LM1 experiment results
# ============================================================

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "mms_kenlm"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

summary_path = (
    RESULTS_DIR
    / "lm1_decoder_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("Saved:")
print(summary_path)

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_kenlm/lm1_decoder_summary.csv


In [ ]:
# ============================================================
# CELL 44: Save detailed LM1 comparison
# ============================================================

# Best LM1 configuration from full validation
best_lm1 = min(
    full_lm_results,
    key=lambda x: x["wer"]
)

best_lm1_predictions = best_lm1["predictions"]

comparison_df = manifest_df[
    ["segment_id", "speaker_group_id", "reference"]
].copy()

comparison_df["greedy_prediction"] = [
    x["prediction_normalized"]
    for x in greedy_results
]

comparison_df["plain_beam_prediction"] = [
    x["prediction_normalized"]
    for x in beam_results
]

comparison_df["lm1_prediction"] = best_lm1_predictions

comparison_df["lm1_alpha"] = best_lm1["alpha"]
comparison_df["lm1_beta"] = best_lm1["beta"]

DETAIL_PATH = (
    PROJECT_ROOT
    / "results"
    / "mms_kenlm"
    / "lm1_detailed_predictions.csv"
)

comparison_df.to_csv(
    DETAIL_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved:")
print(DETAIL_PATH)

print("\nBest LM1 configuration:")
print("alpha =", best_lm1["alpha"])
print("beta  =", best_lm1["beta"])
print("WER   =", round(best_lm1["wer"], 2))
print("CER   =", round(best_lm1["cer"], 2))

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_kenlm/lm1_detailed_predictions.csv

Best LM1 configuration:
alpha = 0.2
beta  = 0.0
WER   = 84.77
CER   = 42.19
